# Flood-It! retention — baseline model experiment

Run the cells in order. Upload the original `flood_it_audit.zip` when asked. This notebook uses the saved project code as of the baseline implementation. It does not connect to BigQuery or require Google Drive permission. The final 790 test rows are not used for model comparison. The download contains **only aggregate results**, model settings and run metadata.


In [ ]:
from pathlib import Path
from tempfile import mkdtemp
import os

# Use a fresh folder for this run. These are the matching files from our GitHub project.
workspace = Path(mkdtemp(prefix='retention_baselines_'))
os.chdir(workspace)
Path("gameplay_ml/__init__.py").parent.mkdir(parents=True, exist_ok=True)
Path("gameplay_ml/__init__.py").write_text("", encoding='utf-8')
Path("gameplay_ml/retention_audit.py").parent.mkdir(parents=True, exist_ok=True)
Path("gameplay_ml/retention_audit.py").write_text("\"\"\"Check the candidate retention table before we train a model.\"\"\"\n\nimport argparse\nimport csv\nfrom collections import Counter\nfrom pathlib import Path\n\n\nFEATURES = [\n    'early_events',\n    'early_sessions',\n    'early_level_starts',\n    'early_level_ends',\n    'early_scores_posted',\n]\nTARGET = 'returned_next_7_days'\nDAY_US = 86_400_000_000\nREQUIRED_COLUMNS = ['user_pseudo_id', 'first_open_us', 'cutoff_us', *FEATURES, TARGET]\n\n\ndef audit_retention(path):\n    players = set()\n    missing = Counter()\n    invalid = Counter()\n    nonzero = Counter()\n    targets = Counter()\n    total = 0\n    duplicates = 0\n\n    with path.open(encoding='utf-8-sig', newline='') as file:\n        reader = csv.DictReader(file)\n        absent = set(REQUIRED_COLUMNS) - set(reader.fieldnames or [])\n        if absent:\n            raise ValueError(f\"Missing columns: {', '.join(sorted(absent))}\")\n\n        for row in reader:\n            total += 1\n            for column in REQUIRED_COLUMNS:\n                if not (row[column] or '').strip():\n                    missing[column] += 1\n\n            # We should only have one row for each player.\n            player = (row['user_pseudo_id'] or '').strip()\n            if player:\n                if player in players:\n                    duplicates += 1\n                players.add(player)\n\n            # Make sure the feature window is exactly 24 hours.\n            try:\n                first_open = int(row['first_open_us'])\n                cutoff = int(row['cutoff_us'])\n                if first_open <= 0 or cutoff - first_open != DAY_US:\n                    invalid['time window'] += 1\n            except (TypeError, ValueError):\n                invalid['time window'] += 1\n\n            for feature in FEATURES:\n                try:\n                    count = int(row[feature])\n                    if count < 0 or (feature == 'early_events' and count == 0):\n                        invalid[feature] += 1\n                    elif count > 0:\n                        nonzero[feature] += 1\n                except (TypeError, ValueError):\n                    invalid[feature] += 1\n\n            # BigQuery may export the Boolean label as true/false or 1/0.\n            label = (row[TARGET] or '').strip().lower()\n            if label in ('true', '1'):\n                targets['returned'] += 1\n            elif label in ('false', '0'):\n                targets['did not return'] += 1\n            else:\n                invalid[TARGET] += 1\n\n    print(f'Rows: {total:,}')\n    print(f'Unique players: {len(players):,}')\n    print(f'Duplicate player rows: {duplicates:,}')\n    print('\\nTarget counts:')\n    for label in ('returned', 'did not return'):\n        print(f'  {label}: {targets[label]:,}')\n    valid_labels = sum(targets.values())\n    if valid_labels:\n        print(f'Return rate: {targets[\"returned\"] / valid_labels:.2%}')\n\n    print('\\nRows with a nonzero feature:')\n    for feature in FEATURES:\n        print(f'  {feature}: {nonzero[feature]:,}')\n\n    print('\\nMissing values:')\n    for column in REQUIRED_COLUMNS:\n        print(f'  {column}: {missing[column]:,}')\n    print('\\nInvalid values:')\n    for column, count in invalid.items():\n        print(f'  {column}: {count:,}')\n    if not invalid:\n        print('  None')\n\n    if not total or duplicates or missing or invalid:\n        raise ValueError('Fix the table issues and run the audit again.')\n    if not targets['returned'] or not targets['did not return']:\n        raise ValueError('We need examples of both outcomes for classification.')\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument('csv_path', type=Path)\n    args = parser.parse_args()\n    try:\n        audit_retention(args.csv_path)\n    except (OSError, ValueError, csv.Error) as error:\n        parser.exit(1, f'Audit failed: {error}\\n')\n\n\nif __name__ == '__main__':\n    main()\n", encoding='utf-8')
Path("gameplay_ml/prepare_retention.py").parent.mkdir(parents=True, exist_ok=True)
Path("gameplay_ml/prepare_retention.py").write_text("\"\"\"Prepare chronological train/test sets for the retention model.\"\"\"\n\nimport argparse\nfrom bisect import bisect_left, bisect_right\nimport csv\nfrom datetime import datetime, timezone\nimport hashlib\nimport json\nfrom pathlib import Path\nfrom zipfile import ZipFile\n\nfrom gameplay_ml.retention_audit import DAY_US, FEATURES, REQUIRED_COLUMNS, TARGET\n\n\nLABEL_WINDOW = 7 * DAY_US\n\n\ndef load_players(path):\n    # Accept the Colab export directly so we do not need another download.\n    if path.suffix.lower() == '.zip':\n        with ZipFile(path) as archive:\n            lines = archive.read('retention_candidates.csv').decode('utf-8-sig').splitlines()\n    else:\n        lines = path.read_text(encoding='utf-8-sig').splitlines()\n\n    reader = csv.DictReader(lines)\n    if set(REQUIRED_COLUMNS) - set(reader.fieldnames or []):\n        raise ValueError('The retention table is missing required columns.')\n\n    players = []\n    seen = set()\n    for raw in reader:\n        player = raw['user_pseudo_id'].strip()\n        if not player or player in seen:\n            raise ValueError('Missing or duplicate player ID.')\n        seen.add(player)\n        row = {'user_pseudo_id': player}\n        for name in ['first_open_us', 'cutoff_us', *FEATURES]:\n            row[name] = int(raw[name])\n        if row['first_open_us'] <= 0 or row['cutoff_us'] - row['first_open_us'] != DAY_US:\n            raise ValueError('Invalid 24-hour feature window.')\n        if any(row[name] < 0 for name in FEATURES):\n            raise ValueError('Negative feature count.')\n        # Total early events include first_open as well as the specific events.\n        if sum(row[name] for name in FEATURES[1:]) >= row['early_events']:\n            raise ValueError('Specific event counts cannot account for all early events.')\n        label = raw[TARGET].strip().lower()\n        if label not in ('true', 'false', '1', '0'):\n            raise ValueError('Invalid retention label.')\n        row[TARGET] = int(label in ('true', '1'))\n        players.append(row)\n\n    # Keep players who have the same prediction time together.\n    return sorted(players, key=lambda row: (row['cutoff_us'], row['user_pseudo_id']))\n\n\ndef require_both_classes(rows, name):\n    if {row[TARGET] for row in rows} != {0, 1}:\n        raise ValueError(f'{name} needs examples of both outcomes.')\n\n\ndef split_players(players, test_fraction=0.2):\n    if not 0 < test_fraction < 1 or len(players) < 2:\n        raise ValueError('Need a nonempty chronological training and testing split.')\n    times = [row['cutoff_us'] for row in players]\n    # Use the newest 20% for testing. Move timestamp ties into the test set.\n    index = int(len(players) * (1 - test_fraction))\n    start = bisect_left(times, times[index])\n    # Only train on players whose entire label window has ended before testing.\n    end = bisect_right(times, times[start] - LABEL_WINDOW, hi=start)\n    train, gap, test = players[:end], players[end:start], players[start:]\n    require_both_classes(train, 'Training set')\n    require_both_classes(test, 'Test set')\n    return train, gap, test\n\n\ndef validation_folds(train, n_folds=5):\n    # Use expanding training windows and later validation blocks. Never use test rows.\n    times = [row['cutoff_us'] for row in train]\n    if n_folds < 1 or len(train) <= n_folds:\n        raise ValueError('Not enough training rows for validation.')\n    bounds = [0]\n    for fold in range(1, n_folds + 1):\n        index = len(train) * fold // (n_folds + 1)\n        bounds.append(bisect_left(times, times[index]))\n    bounds.append(len(train))\n\n    folds = []\n    for fold in range(1, n_folds + 1):\n        start, stop = bounds[fold], bounds[fold + 1]\n        end = bisect_right(times, times[start] - LABEL_WINDOW, hi=start)\n        require_both_classes(train[:end], f'Fold {fold} training')\n        require_both_classes(train[start:stop], f'Fold {fold} validation')\n        # The indices are zero-based; the end index is not included.\n        folds.append((end, start, stop))\n    return folds\n\n\ndef summarize(rows):\n    returned = sum(row[TARGET] for row in rows)\n    return {'players': len(rows), 'returned': returned, 'did_not_return': len(rows) - returned}\n\n\ndef write_csv(path, rows):\n    with path.open('w', encoding='utf-8', newline='') as output:\n        writer = csv.DictWriter(output, fieldnames=REQUIRED_COLUMNS)\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef prepare(source, output):\n    players = load_players(source)\n    train, gap, test = split_players(players)\n    folds = validation_folds(train)\n    summary = {\n        'source_sha256': hashlib.sha256(source.read_bytes()).hexdigest(),\n        'features': FEATURES,\n        'target': TARGET,\n        'label_window_days': 7,\n        'first_test_prediction_utc': datetime.fromtimestamp(test[0]['cutoff_us'] / 1e6, timezone.utc).isoformat(),\n        'all_players': summarize(players),\n        'train': summarize(train),\n        'gap': summarize(gap),\n        'test': summarize(test),\n        'folds': [\n            {'fold': number, 'train_end': end, 'validation_start': start, 'validation_end': stop,\n             'train': summarize(train[:end]), 'validation': summarize(train[start:stop])}\n            for number, (end, start, stop) in enumerate(folds, 1)\n        ],\n    }\n    # Keep player-level files local, and do not overwrite an earlier experiment.\n    output.mkdir(parents=True, exist_ok=False)\n    write_csv(output / 'train.csv', train)\n    write_csv(output / 'test.csv', test)\n    (output / 'split_summary.json').write_text(json.dumps(summary, indent=2) + '\\n', encoding='utf-8')\n    print(json.dumps(summary, indent=2))\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument('source', type=Path)\n    parser.add_argument('--output', type=Path, default=Path('data/processed/retention_split'))\n    args = parser.parse_args()\n    prepare(args.source, args.output)\n\n\nif __name__ == '__main__':\n    main()\n", encoding='utf-8')
Path("gameplay_ml/train_baselines.py").parent.mkdir(parents=True, exist_ok=True)
Path("gameplay_ml/train_baselines.py").write_text("\"\"\"Compare untuned retention models on the five training-data folds.\"\"\"\n\nimport argparse\nimport csv\nfrom datetime import datetime, timezone\nimport hashlib\nimport json\nimport platform\nfrom importlib.metadata import version\nfrom pathlib import Path\nfrom statistics import mean, stdev\nfrom time import perf_counter\n\nimport numpy as np\nfrom sklearn.dummy import DummyClassifier\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import (\n    accuracy_score, average_precision_score, balanced_accuracy_score,\n    brier_score_loss, f1_score, precision_score, recall_score, roc_auc_score,\n)\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom threadpoolctl import threadpool_limits\nfrom xgboost import XGBClassifier\n\nfrom gameplay_ml.prepare_retention import LABEL_WINDOW, load_players, summarize, validation_folds\nfrom gameplay_ml.retention_audit import FEATURES, TARGET\n\n\nSEED = 42\nMETRICS = ['average_precision', 'roc_auc', 'accuracy', 'balanced_accuracy',\n           'precision', 'recall', 'f1', 'brier_score']\n\n\ndef models():\n    # Fixed starting settings, before any hyperparameter search.\n    dummy = {'strategy': 'prior'}\n    logistic = {'C': 1.0, 'solver': 'lbfgs', 'max_iter': 1000, 'random_state': SEED}\n    forest = {'n_estimators': 100, 'random_state': SEED, 'n_jobs': 1}\n    boosted = {'n_estimators': 100, 'max_depth': 6, 'learning_rate': 0.3,\n               'objective': 'binary:logistic', 'eval_metric': 'logloss',\n               'random_state': SEED, 'n_jobs': 1, 'tree_method': 'hist'}\n    return [\n        ('Dummy', DummyClassifier(**dummy), dummy),\n        ('Logistic Regression', make_pipeline(StandardScaler(), LogisticRegression(**logistic)),\n         {'preprocessing': 'StandardScaler fitted within each fold', **logistic}),\n        ('Random Forest', RandomForestClassifier(**forest), forest),\n        ('XGBoost', XGBClassifier(**boosted), boosted),\n    ]\n\n\ndef score_predictions(labels, probabilities):\n    # Class 1 means the player returned. Use the same threshold for every model.\n    predictions = (probabilities >= 0.5).astype(int)\n    return {\n        'average_precision': float(average_precision_score(labels, probabilities)),\n        'roc_auc': float(roc_auc_score(labels, probabilities)),\n        'accuracy': float(accuracy_score(labels, predictions)),\n        'balanced_accuracy': float(balanced_accuracy_score(labels, predictions)),\n        'precision': float(precision_score(labels, predictions, zero_division=0)),\n        'recall': float(recall_score(labels, predictions, zero_division=0)),\n        'f1': float(f1_score(labels, predictions, zero_division=0)),\n        'brier_score': float(brier_score_loss(labels, probabilities)),\n    }\n\n\ndef run_baselines(split_dir, output):\n    # Do not load test.csv. It is reserved for the final experiment.\n    train_file = split_dir / 'train.csv'\n    train = load_players(train_file)\n    summary = json.loads((split_dir / 'split_summary.json').read_text())\n    if summary['features'] != FEATURES or summary['target'] != TARGET:\n        raise ValueError('The split uses a different feature set or target.')\n    if summary['train'] != summarize(train):\n        raise ValueError('The training rows do not match the split summary.')\n    cutoff = datetime.fromisoformat(summary['first_test_prediction_utc'])\n    if train[-1]['cutoff_us'] + LABEL_WINDOW > round(cutoff.timestamp() * 1e6):\n        raise ValueError('Training labels overlap the test prediction window.')\n    folds = validation_folds(train)\n    expected = [(f['train_end'], f['validation_start'], f['validation_end'])\n                for f in summary['folds']]\n    if folds != expected:\n        raise ValueError('Validation fold boundaries do not match the saved split.')\n\n    X = np.array([[row[name] for name in FEATURES] for row in train], dtype=float)\n    y = np.array([row[TARGET] for row in train], dtype=int)\n    settings = {name: params for name, _, params in models()}\n    results = []\n    for number, (end, start, stop) in enumerate(folds, 1):\n        for name, model, _ in models():\n            # Build a fresh model each time. Preprocessing is fitted only on training rows.\n            with threadpool_limits(limits=1):\n                began = perf_counter()\n                model.fit(X[:end], y[:end])\n                seconds = perf_counter() - began\n                probabilities = model.predict_proba(X[start:stop])[:, 1]\n            scores = score_predictions(y[start:stop], probabilities)\n            results.append({'model': name, 'fold': number, 'train_rows': end,\n                            'validation_rows': stop - start,\n                            'validation_return_rate': float(y[start:stop].mean()),\n                            'fit_seconds': seconds, **scores})\n            print(f\"{name:20s} fold {number}: AP={scores['average_precision']:.4f}\")\n\n    comparisons = []\n    for name in settings:\n        model_rows = [row for row in results if row['model'] == name]\n        item = {'model': name}\n        for metric in [*METRICS, 'fit_seconds']:\n            values = [row[metric] for row in model_rows]\n            item[metric + '_mean'] = mean(values)\n            item[metric + '_std'] = stdev(values)\n        comparisons.append(item)\n    comparisons.sort(key=lambda row: row['average_precision_mean'], reverse=True)\n\n    # Everything saved here is aggregate. No IDs, labels or individual predictions.\n    output.mkdir(parents=True, exist_ok=False)\n    for filename, rows in [('fold_metrics.csv', results), ('model_comparison.csv', comparisons)]:\n        with (output / filename).open('w', newline='', encoding='utf-8') as file:\n            writer = csv.DictWriter(file, fieldnames=list(rows[0]))\n            writer.writeheader()\n            writer.writerows(rows)\n    metadata = {\n        'run_utc': datetime.now(timezone.utc).isoformat(),\n        'experiment': 'retention_untuned_baselines',\n        'python': platform.python_version(),\n        'platform': platform.platform(),\n        'packages': {name: version(name) for name in\n                     ['numpy', 'scipy', 'scikit-learn', 'xgboost', 'threadpoolctl']},\n        'seed': SEED,\n        'cpu_threads': 1,\n        'source_sha256': summary['source_sha256'],\n        'train_csv_sha256': hashlib.sha256(train_file.read_bytes()).hexdigest(),\n        'features': FEATURES,\n        'target': TARGET,\n        'folds': summary['folds'],\n        'primary_metric': 'average_precision',\n        'classification_threshold': 0.5,\n        'model_settings': settings,\n        'test_used': False,\n    }\n    (output / 'experiment.json').write_text(json.dumps(metadata, indent=2) + '\\n')\n    print('\\nValidation comparison (mean average precision):')\n    for row in comparisons:\n        print(f\"{row['model']:20s} {row['average_precision_mean']:.4f} \"\n              f\"(SD {row['average_precision_std']:.4f})\")\n    return comparisons\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument('--split-dir', type=Path, default=Path('data/processed/retention_split'))\n    parser.add_argument('--output', type=Path, default=Path('results/retention_baselines'))\n    args = parser.parse_args()\n    run_baselines(args.split_dir, args.output)\n\n\nif __name__ == '__main__':\n    main()\n", encoding='utf-8')
Path("tests/test_prepare_retention.py").parent.mkdir(parents=True, exist_ok=True)
Path("tests/test_prepare_retention.py").write_text("import csv\nfrom pathlib import Path\nfrom tempfile import TemporaryDirectory\nimport unittest\n\nfrom gameplay_ml.prepare_retention import (\n    DAY_US, LABEL_WINDOW, TARGET, load_players, split_players, validation_folds,\n)\nfrom gameplay_ml.retention_audit import REQUIRED_COLUMNS\n\n\ndef player(number, day, label):\n    return {\n        'user_pseudo_id': str(number),\n        'first_open_us': day * DAY_US,\n        'cutoff_us': (day + 1) * DAY_US,\n        'early_events': 4,\n        'early_sessions': 1,\n        'early_level_starts': 1,\n        'early_level_ends': 1,\n        'early_scores_posted': 0,\n        TARGET: label,\n    }\n\n\nclass PrepareRetentionTests(unittest.TestCase):\n    def test_train_test_gap_and_ties(self):\n        rows = [player(i, i + 1, i % 2) for i in range(100)]\n        # Make two players share the first test timestamp.\n        rows[79]['first_open_us'] = rows[80]['first_open_us']\n        rows[79]['cutoff_us'] = rows[80]['cutoff_us']\n        train, gap, test = split_players(rows)\n        self.assertEqual(len(train) + len(gap) + len(test), len(rows))\n        self.assertEqual(len(test), 21)\n        self.assertLessEqual(train[-1]['cutoff_us'] + LABEL_WINDOW, test[0]['cutoff_us'])\n        self.assertGreater(gap[0]['cutoff_us'] + LABEL_WINDOW, test[0]['cutoff_us'])\n\n    def test_five_validation_folds(self):\n        rows = [player(i, i + 1, i % 2) for i in range(150)]\n        folds = validation_folds(rows)\n        self.assertEqual(len(folds), 5)\n        for end, start, stop in folds:\n            self.assertLess(end, start)\n            self.assertLess(start, stop)\n            self.assertLessEqual(rows[end - 1]['cutoff_us'] + LABEL_WINDOW, rows[start]['cutoff_us'])\n            self.assertEqual({row[TARGET] for row in rows[start:stop]}, {0, 1})\n        self.assertEqual(folds[-1][2], len(rows))\n\n    def test_load_and_validate(self):\n        with TemporaryDirectory() as folder:\n            path = Path(folder) / 'players.csv'\n            rows = [player(1, 2, 'True'), player(2, 1, 'False')]\n            with path.open('w', newline='') as output:\n                writer = csv.DictWriter(output, fieldnames=REQUIRED_COLUMNS)\n                writer.writeheader()\n                writer.writerows(rows)\n            loaded = load_players(path)\n            self.assertEqual([row[TARGET] for row in loaded], [0, 1])\n            with path.open('a', newline='') as output:\n                csv.DictWriter(output, fieldnames=REQUIRED_COLUMNS).writerow(rows[0])\n            with self.assertRaisesRegex(ValueError, 'duplicate'):\n                load_players(path)\n\n    def test_reject_invalid_time_window(self):\n        with TemporaryDirectory() as folder:\n            path = Path(folder) / 'players.csv'\n            row = player(1, 4, 1)\n            row['cutoff_us'] += 1\n            with path.open('w', newline='') as output:\n                writer = csv.DictWriter(output, fieldnames=REQUIRED_COLUMNS)\n                writer.writeheader()\n                writer.writerow(row)\n            with self.assertRaisesRegex(ValueError, '24-hour'):\n                load_players(path)\n\n\nif __name__ == '__main__':\n    unittest.main()\n", encoding='utf-8')
Path("tests/test_train_baselines.py").parent.mkdir(parents=True, exist_ok=True)
Path("tests/test_train_baselines.py").write_text("\"\"\"Small tests for the baseline experiment, using synthetic data only.\"\"\"\nimport csv\nimport json\nfrom pathlib import Path\nfrom tempfile import TemporaryDirectory\nimport unittest\n\nimport numpy as np\n\nfrom gameplay_ml.retention_audit import DAY_US, REQUIRED_COLUMNS, TARGET\nfrom gameplay_ml.prepare_retention import prepare\nfrom gameplay_ml.train_baselines import run_baselines, score_predictions\n\n\nclass BaselineTests(unittest.TestCase):\n    def test_perfect_predictions(self):\n        scores = score_predictions(np.array([0, 1, 0, 1]), np.array([0.1, 0.8, 0.2, 0.9]))\n        for metric in ('average_precision', 'roc_auc', 'accuracy', 'precision', 'recall', 'f1'):\n            self.assertEqual(scores[metric], 1.0)\n\n    def test_all_models_without_test_file(self):\n        with TemporaryDirectory() as folder:\n            root = Path(folder)\n            source = root / 'synthetic.csv'\n            with source.open('w', newline='') as file:\n                writer = csv.DictWriter(file, fieldnames=REQUIRED_COLUMNS)\n                writer.writeheader()\n                for index in range(150):\n                    label = index % 2\n                    writer.writerow({\n                        'user_pseudo_id': str(index),\n                        'first_open_us': (index + 1) * DAY_US,\n                        'cutoff_us': (index + 2) * DAY_US,\n                        'early_events': 5 + label,\n                        'early_sessions': 1, 'early_level_starts': 1,\n                        'early_level_ends': 1, 'early_scores_posted': label,\n                        TARGET: label,\n                    })\n            split = root / 'split'\n            prepare(source, split)\n            # The baseline script must work without the held-out data.\n            (split / 'test.csv').unlink()\n            output = root / 'results'\n            comparison = run_baselines(split, output)\n            self.assertEqual(len(comparison), 4)\n            with (output / 'fold_metrics.csv').open(newline='') as file:\n                self.assertEqual(len(list(csv.DictReader(file))), 20)\n            self.assertFalse(json.loads((output / 'experiment.json').read_text())['test_used'])\n\n\nif __name__ == '__main__':\n    unittest.main()\n", encoding='utf-8')
Path("requirements-baselines.txt").parent.mkdir(parents=True, exist_ok=True)
Path("requirements-baselines.txt").write_text("numpy==2.2.6\nscikit-learn==1.7.2\nxgboost==3.0.5\n", encoding='utf-8')


In [ ]:
# Install the versions we are using for this experiment.
%pip install -q -r requirements-baselines.txt


In [ ]:
from google.colab import files
import subprocess
import sys

uploaded = files.upload()
if set(uploaded) != {'flood_it_audit.zip'}:
    raise ValueError('Upload only the original flood_it_audit.zip file.')
Path('flood_it_audit.zip').write_bytes(uploaded['flood_it_audit.zip'])
del uploaded

# Validate the code with synthetic data before running the real experiment.
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)


In [ ]:
# Reproduce the chronological training split, then run five-fold validation.
subprocess.run([sys.executable, '-m', 'gameplay_ml.prepare_retention', 'flood_it_audit.zip'], check=True)
subprocess.run([sys.executable, '-m', 'gameplay_ml.train_baselines'], check=True)


In [ ]:
# Download only the aggregate experiment results and settings.
import shutil
archive = shutil.make_archive('retention_baseline_results', 'zip', root_dir='results/retention_baselines')
files.download(archive)


Upload `retention_baseline_results.zip` to our project chat. Do not upload or commit individual player data to GitHub.